# 03 · Training (baseline EfficientNet-B0, primary EfficientNetV2-S)
> **Research prototype.** Predicts *dataset-defined* dementia categories from preprocessed JPG brain MRI slices. Not a clinically validated diagnostic system.

Steps 17–19. Both models use exactly the same split, preprocessing, augmentation, metrics and two-stage strategy:
stage 1 frozen backbone (head LR 1e-3), stage 2 last ~35% of backbone parameters unfrozen (backbone LR 1e-5, head LR
1e-4), AdamW, ReduceLROnPlateau and early stopping on **validation macro-F1**. The test set is not used.
Re-running the cell resumes from `last_checkpoint.pth`. CUDA out-of-memory halves the batch size and retries.

In [ ]:
# Bootstrap: locate the project code (local checkout, /kaggle/working, or a Kaggle dataset upload)
import os, sys, glob, shutil, json
from pathlib import Path

CANDIDATES = [Path.cwd().parent, Path.cwd(), Path("/kaggle/working/dementia_mri_project")]
CANDIDATES += [Path(p).parent for p in glob.glob("/kaggle/input/*/dementia_mri_project/src")
               + glob.glob("/kaggle/input/*/*/dementia_mri_project/src") + glob.glob("/kaggle/input/*/src")]
CODE = next((c for c in CANDIDATES if (c / "src" / "run_pipeline.py").exists()), None)
if CODE is None:
    raise FileNotFoundError("Project code not found. On Kaggle: upload dementia_mri_project.zip as a Dataset and add it "
                            "as an Input (Kaggle extracts it), or put the project folder next to this notebook.")
if str(CODE).startswith("/kaggle/input"):                       # /kaggle/input is read-only -> work in /kaggle/working
    dst = Path("/kaggle/working/dementia_mri_project")
    shutil.copytree(CODE / "src", dst / "src", dirs_exist_ok=True)
    shutil.copy(CODE / "predict.py", dst / "predict.py")
    if not (dst / "configs" / "config.json").exists():
        shutil.copytree(CODE / "configs", dst / "configs", dirs_exist_ok=True)
    for sub in ("data", "checkpoints", "results", "gradcam", "reports"):   # outputs of earlier notebooks, if attached
        if (CODE / sub).exists() and not (dst / sub).exists():
            shutil.copytree(CODE / sub, dst / sub)
    CODE = dst
sys.path.insert(0, str(CODE)); os.chdir(CODE)
print("Project code:", CODE)

from src.utils import load_config, get_paths, print_env, load_json, save_json
from src.nb_helpers import show, show_md, show_table, sample_grid

CONFIG_OVERRIDES = {}      # e.g. {"batch_size": 16, "num_workers": 4, "epochs_stage2": 30}
cfg = load_config(overrides=CONFIG_OVERRIDES)
paths = get_paths(cfg)
ENV = print_env()
print("Outputs ->", paths["root"])

In [ ]:
import torch
print("GPU:", ENV.get("gpu"), "| memory GB:", ENV.get("gpu_memory_gb"), "| CUDA:", ENV.get("cuda_available"))
if ENV.get("gpu_memory_gb") and ENV["gpu_memory_gb"] < 10 and cfg["batch_size"] > 32:
    print("!! < 10 GB GPU memory: consider CONFIG_OVERRIDES = {'batch_size': 24}")
split = load_json(paths["data"] / "split_summary.json")
assert split is not None, "run notebook 02 first"
print("Leakage checks:", split["leakage_checks"])

## Baseline: EfficientNet-B0

In [ ]:
from src.run_pipeline import stage_train
RUNS = stage_train(cfg, paths, which=("baseline",))

In [ ]:
show(paths["results"] / "runs" / "baseline" / "training_curves.png")

## Primary: EfficientNetV2-S

In [ ]:
RUNS = stage_train(cfg, paths, which=("primary",))
print(json.dumps(RUNS, indent=1, default=str))

In [ ]:
for f in ("training_loss.png", "training_accuracy.png", "training_macro_f1.png", "learning_rate.png"):
    show(paths["results"] / f, width=600)
import pandas as pd
display(pd.read_csv(paths["results"] / "training_history.csv").round(4))